In [1]:
!pip install git+https://github.com/florencejt/fusilli.git
!pip install nibabel

  Cloning https://github.com/florencejt/fusilli.git to c:\users\bnish\appdata\local\temp\pip-req-build-wf28_i33
  Resolved https://github.com/florencejt/fusilli.git to commit bbd29f94f9ec43c22d8e15e87a05b86fad25752f
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/florencejt/fusilli.git 'C:\Users\bnish\AppData\Local\Temp\pip-req-build-wf28_i33'
  Running command git submodule update --init --recursive -q


In [49]:
import os
import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch import tensor
import numpy as np
import nibabel as nib
import glob

from fusilli.data import prepare_fusion_data
from fusilli.train import train_and_save_models
from fusilli.fusionmodels.tabularimagefusion.concat_img_maps_tabular_maps import ConcatImageMapsTabularMaps
from fusilli.eval import RealsVsPreds, ConfusionMatrix

In [3]:
pip show torch

Name: torch
Version: 2.6.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: C:\Users\bnish\anaconda3\Lib\site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: fusilli, lightning, pytorch-lightning, torchmetrics, torchvision
Note: you may need to restart the kernel to use updated packages.


In [4]:
#print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))
import sys
import torch
print("Python version:", sys.version)
# print("Torch version:", torch._version_)
print("Torch CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
# print("TorchMetrics version:", torchmetrics._version_)
# print("Lightning version:", lightning._version_)
# print("WandB version:", wandb._version_)
# print("Fusilli version:", fusilli._version_)


Python version: 3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)]
Torch CUDA available: False
CUDA device: CPU only


In [51]:
## Data preprocessing
print('cwd (base):', os.getcwd(),'\n')

# open and load csv
base_dir = Path(r"C:\Users\bnish\Downloads\ATLAS_R2.0\ATLAS_R2.0\ATLAS_2")  
meta_xlsx  = base_dir / "20220425_ATLAS_2.0_MetaData.xlsx"          
train_dir  = base_dir / "Training"  
test_dir   = base_dir / "Testing" 


meta_df = pd.read_excel(meta_xlsx)

cwd (base): C:\Users\bnish\Downloads\ATLAS_R2.0\ATLAS_R2.0 



In [53]:
from pathlib import Path
import re
import pandas as pd


nii_paths = list(train_dir.rglob("*.nii*")) + list(test_dir.rglob("*.nii*"))
print(f"Found {len(nii_paths)} NIfTI files under Training/Testing")

def is_structural(p: Path) -> bool:
    s = p.name.lower()
    if any(bad in s for bad in ["mask", "lesion", "seg", "label", "prob", "wmh", "flair", "dwi"]):
        return False
    return ("t1w" in s) or ("t1" in s and "t2" not in s)

SUBJ_RE = re.compile(r"(sub-[a-zA-Z0-9]+s[0-9]+)")
def extract_subj_id(p: Path):
    m = SUBJ_RE.search(str(p).replace("\\", "/"))
    return m.group(1) if m else None

by_subj = {}
for p in nii_paths:
    sid = extract_subj_id(p)
    if sid is None:
        continue
    if not is_structural(p):
        continue
    # Prefer explicit T1w over generic T1
    cur = by_subj.get(sid)
    if cur is None:
        by_subj[sid] = p
    else:
        if "t1w" in p.name.lower() and "t1w" not in cur.name.lower():
            by_subj[sid] = p

print(f"Indexed {len(by_subj)} subjects with structural images")

def map_path(sid):
    sid = str(sid).strip()
    return str(by_subj.get(sid)) if sid in by_subj else None

meta_df["image_path"] = meta_df["Subject ID"].apply(map_path)
missing = meta_df[meta_df["image_path"].isna()]["Subject ID"].tolist()
print(f"Resolved NIfTI paths for {meta_df['image_path'].notna().sum()} subjects")
if missing:
    print("Could not resolve for (first 10):", missing[:10])

meta_df_paths = meta_df[meta_df["image_path"].notna()].reset_index(drop=True)

Found 1610 NIfTI files under Training/Testing
Indexed 955 subjects with structural images
Resolved NIfTI paths for 655 subjects


In [55]:
def map_hemi(x):
    x = str(x).strip().lower()
    if x.startswith("l"): return 0
    if x.startswith("r"): return 1
    return 2
meta_df_paths["prediction_label"] = meta_df_paths["Primary Stroke Hemisphere"].apply(map_hemi)

# Split using actual folder roots
meta_df_paths["image_path"] = meta_df_paths["image_path"].astype(str)

df_train = meta_df_paths[
    meta_df_paths["image_path"].str.contains(str(train_dir), regex=False, na=False)
].reset_index(drop=True)

df_val = meta_df_paths[
    meta_df_paths["image_path"].str.contains(str(test_dir), regex=False, na=False)
].reset_index(drop=True)

if len(df_val) < 5:
    from sklearn.model_selection import StratifiedGroupKFold
    gkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    y = df_train["prediction_label"].values
    groups = df_train["Subject ID"].values
    tr_idx, va_idx = next(gkf.split(df_train, y, groups))
    df_val   = df_train.iloc[va_idx].reset_index(drop=True)
    df_train = df_train.iloc[tr_idx].reset_index(drop=True)


In [59]:
def norm_sub_id(x):
    s = str(x).strip()
    return s.replace("sub-","")

meta_df["ID"] = meta_df["Subject ID"].apply(norm_sub_id)

In [7]:
# # T1w image paths
# def find_t1_for_subject(sub_id: str) -> str | None:
#     patterns = [
#         str(train_dir / f"**/*{sub_id}*/*anat/*T1w*.nii*"),
#         str(train_dir / f"**/*{sub_id}*/*T1w*.nii*"),
#         str(train_dir / f"**/*{sub_id}*T1w*.nii*"),
#     ]
#     for pat in patterns:
#         hits = glob.glob(pat, recursive=True)
#         if hits:
#             return hits[0]
#     return None

# meta_df["image_path"] = meta_df["ID"].apply(find_t1_for_subject)

# before = len(meta_df)
# meta_df = meta_df.dropna(subset=["image_path"]).reset_index(drop=True)
# after = len(meta_df)
# print(f"Fixed T1w paths for {after}/{before} subjects in Training")

Fixed T1w paths for 655/655 subjects in Training


In [61]:
# Add labels
if "Primary Stroke Hemisphere" in meta_df.columns:
    meta_df["prediction_label"] = meta_df["Primary Stroke Hemisphere"].apply(map_hemi)
    
else:
    if "Lesion Volume" not in meta_df.columns:
        raise ValueError("Neither 'Primary Stroke Hemisphere' nor 'Lesion Volume' in MetaData.xlsx.")
    q = meta_df["Lesion Volume"].quantile([0.33, 0.66]).values
    
    def size_bin(v):
        if v <= q[0]: return 0
        if v <= q[1]: return 1
        return 2
        
    meta_df["prediction_label"] = meta_df["Lesion Volume"].apply(size_bin)
    print("Using 3-class lesion size bins from 'Lesion Volume'.")

In [63]:
# New csv for mapped data for later
mapped_csv = base_dir / "mapped_atlas_metadata.csv"
meta_df.to_csv(mapped_csv, index=False)
print(f"Saved mapped metadata in {mapped_csv}...\n")
print(meta_df[["ID","image_path","Primary Stroke Hemisphere" if "Primary Stroke Hemisphere" in meta_df.columns else "Lesion Volume"]].head())

Saved mapped metadata in C:\Users\bnish\Downloads\ATLAS_R2.0\ATLAS_R2.0\ATLAS_2\mapped_atlas_metadata.csv...

         ID                                         image_path  \
0  r001s001  C:\Users\bnish\Downloads\ATLAS_R2.0\ATLAS_R2.0...   
1  r001s002  C:\Users\bnish\Downloads\ATLAS_R2.0\ATLAS_R2.0...   
2  r001s003  C:\Users\bnish\Downloads\ATLAS_R2.0\ATLAS_R2.0...   
3  r001s004  C:\Users\bnish\Downloads\ATLAS_R2.0\ATLAS_R2.0...   
4  r001s005  C:\Users\bnish\Downloads\ATLAS_R2.0\ATLAS_R2.0...   

  Primary Stroke Hemisphere  
0                     Right  
1                     Right  
2                     Right  
3                      Left  
4                      Left  


In [99]:
import numpy as np, torch, nibabel as nib
from torch.utils.data import Dataset

class ScanMILDatasetWithTab(Dataset):
    def __init__(self, df, img_col, label_col, tab_cols,
                 target_size=224, clip_percentiles=(0.5,99.5),
                 slice_stride=2, max_slices=128):
        self.df = df.reset_index(drop=True)
        self.img_col = img_col
        self.label_col = label_col
        self.tab_cols = list(tab_cols) if tab_cols is not None else []
        self.target_size = target_size
        self.clip_percentiles = clip_percentiles
        self.slice_stride = max(1, int(slice_stride))
        self.max_slices = max_slices

    def __len__(self): return len(self.df)

    @staticmethod
    def _zscore(x):
        m, s = np.mean(x), np.std(x)+1e-8
        return (x-m)/s

    def __getitem__(self, idx):
        row = self.df.loc[idx]
        path = str(row[self.img_col])
        y = int(row[self.label_col])
        if len(self.tab_cols) > 0:
            tab_np = row[self.tab_cols].to_numpy(dtype=np.float32, copy=False)
            tab = torch.from_numpy(tab_np)
        else:
            tab = torch.zeros(0, dtype=torch.float32)

        nii = nib.load(path, mmap=True)
        vol = nib.as_closest_canonical(nii).get_fdata(dtype=np.float32)   

        vol = vol[..., ::self.slice_stride]
        S = vol.shape[2]
        if self.max_slices and S > self.max_slices:
            step = S / self.max_slices
            idxs = (np.arange(self.max_slices) * step).astype(int)
            vol = vol[..., idxs]; S = vol.shape[2]

        lo, hi = np.percentile(vol, self.clip_percentiles)
        vol = np.clip(vol, lo, hi)
        vol = self._zscore(vol)

        v = torch.from_numpy(vol.transpose(2,0,1)).unsqueeze(1).float()   
        import torch.nn.functional as F
        v = v.permute(1,0,2,3)                                           
        v = F.interpolate(v, size=(self.target_size, self.target_size),
                          mode="bilinear", align_corners=False)           
        v = v.permute(1,0,2,3).contiguous()                               

        return v, torch.tensor(y), tab, {"path": path, "num_slices": S}


In [13]:
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.Grayscale(num_output_channels=1),
#     transforms.ToTensor()
# ])

# # def middle_axial_slice_2uint8(nifti_path: Path) -> Image.Image:
# #     ni = nib.load(str(nifti_path))
# #     data = ni.get_fdata()
# #     z = data.shape[2] // 2
# #     slice2d = data[:, :, z]
# #     lo, hi = np.percentile(slice2d[np.isfinite(slice2d)], [1, 99])
# #     slice2d = np.clip((slice2d - lo) / (hi - lo + 1e-8), 0, 1)
# #     slice2d = (slice2d * 255.0).astype(np.uint8)
# #     return Image.fromarray(slice2d)  # single-channel

# # df = pd.read_csv(mapped_csv)
# # images, valid_idx = [], []
# # for i, row in tqdm(df.iterrows(), total=len(df)):
# #     p = Path(row["image_path"])
# #     if not p.exists(): continue
# #     pil_img = middle_axial_slice_2uint8(p)
# #     images.append(transform(pil_img))   # [1, 224, 224]
# #     valid_idx.append(i)

# # image_tensor = torch.stack(images)      # [N, 1, 224, 224]
# # torch.save(image_tensor, base_dir / "images.pt")
# # df = df.loc[valid_idx].reset_index(drop=True)
# # df.to_csv(mapped_csv, index=False)
# # print("Saved image tensor:", base_dir / "images.pt", "shape:", image_tensor.shape)

In [67]:
df_numeric = pd.read_csv(mapped_csv)
df_numeric["prediction_label"] = pd.to_numeric(df_numeric["prediction_label"], errors="coerce").astype("Int64")
non_feature_cols = {"ID", "prediction_label"}

def to_numeric_series(s: pd.Series) -> pd.Series:
    if s.dtype.kind in "biufc":
        return s
    str_s = s.astype(str).str.strip()
    first_number = str_s.str.extract(r"([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)", expand=False)
    out = pd.to_numeric(first_number, errors="coerce")
    return out

clean = {}
for c in df_numeric.columns:
    if c in non_feature_cols:
        clean[c] = df_numeric[c]
    else:
        clean[c] = to_numeric_series(df_numeric[c])

clean = pd.DataFrame(clean)

feature_cols = [c for c in clean.columns if c not in non_feature_cols]
min_non_na = max(10, int(0.6 * len(clean))) 
kept = []
dropped = []
for c in feature_cols:
    if clean[c].notna().sum() >= min_non_na:
        kept.append(c)
    else:
        dropped.append(c)
clean = clean[["ID", "prediction_label"] + kept].dropna().reset_index(drop=True)

clean[kept] = clean[kept].astype("float32")
clean["prediction_label"] = clean["prediction_label"].astype("int64")

tabular_csv = base_dir / "mapped_atlas_numeric.csv"
clean.to_csv(tabular_csv, index=False)

print(f"[Tabular cleanup] kept {len(kept)} numeric feature columns; dropped {len(dropped)}: {dropped}")
print(f"[Tabular cleanup] final shape: {clean.shape}")



data_paths = {
    "tabular1": str(tabular_csv),
    "tabular2": "",
    "image": str(base_dir / "images.pt"),
}

[Tabular cleanup] kept 13 numeric feature columns; dropped 11: ['Organism', 'Organism Part', 'Developmental Stage', 'Primary Stroke Hemisphere', 'Primary Stroke Location', 'Secondary Stroke Hemisphere', 'Secondary Stroke Location', 'Scanner Brand', 'ATLAS 1.2 Subject ID', 'INDI Subject ID', 'Chronicity']
[Tabular cleanup] final shape: (544, 15)


In [ ]:
# import re
# df_numeric = pd.read_csv(mapped_csv)
# df_numeric["prediction_label"] = pd.to_numeric(df_numeric["prediction_label"], errors="coerce").astype("Int64")


# hemi_col   = "Primary Stroke Hemisphere"
# loc_col   = "Primary Stroke Location"
# days_col = "Days Post Stroke"
# selected_cols = [hemi_col, loc_col, days_col] 

# days = pd.to_numeric(df_numeric[days_col], errors="coerce")

# # Hemisphere: Left=0, Right=1, OUnknown=2
# def hemi_encode(x):
#     s = str(x).strip().lower()
#     if s.startswith("l"): return 0  
#     if s.startswith("r"): return 1  
#     return 2

# hemi_code = df_numeric[hemi_col].apply(hemi_encode).astype("float32")

# loc_raw = df_numeric[loc_col].astype(str)

# def normalize_loc_string(s: str) -> list[str]:
#     s = s.strip().lower()
#     if not s or s in {"nan", "none"}:
#         return []
#     # unify separators ; and / into commas
#     s = re.sub(r"[;/]", ",", s)
#     # remove characters that are not letters, commas, or spaces
#     s = re.sub(r"[^a-z, ]", "", s)
#     parts = [p.strip() for p in s.split(",") if p.strip()]
#     # unique tokens per row
#     return sorted(set(parts))

# vocab = set()
# for row in loc_raw:
#     vocab.update(normalize_loc_string(row))
# vocab = sorted(vocab)
# print("Detected stroke location categories:", vocab)

# def slugify(loc_name: str) -> str:
#     slug = re.sub(r"[^a-z0-9]+", "_", loc_name.lower()).strip("_")
#     return slug or "unknown"

# loc_feature_names = []
# for loc in vocab:
#     col_name = f"loc_{slugify(loc)}"
#     loc_feature_names.append(col_name)

#     def has_loc(row_str, loc=loc):
#         tokens = normalize_loc_string(row_str)
#         return float(loc in tokens)

#     df_numeric[col_name] = loc_raw.apply(has_loc).astype("float32")

# clean = pd.DataFrame({
#     "ID": df_numeric["ID"],
#     "prediction_label": df_numeric["prediction_label"],
#     "days_post_stroke": days,
#     "hemi_code": hemi_code,
# })
# for col_name in loc_feature_names:
#     clean[col_name] = df_numeric[col_name]

# clean["prediction_label"] = pd.to_numeric(clean["prediction_label"], errors="coerce")
# clean = clean.dropna(subset=["prediction_label", "days_post_stroke", "hemi_code"]).reset_index(drop=True)
# feature_cols = ["days_post_stroke", "hemi_code"] + loc_feature_names
# clean["prediction_label"] = clean["prediction_label"].astype("int64")
# clean[feature_cols] = clean[feature_cols].astype("float32")


# tabular_csv = base_dir / "mapped_atlas_numeric.csv"
# clean.to_csv(tabular_csv, index=False)

# print(f"[Tabular cleanup] final shape: {clean.shape}")
# print("[Tabular cleanup] example rows:")
# print(clean)

In [ ]:
print(clean)

In [69]:
output_paths = {
    "checkpoints": "outputs/checkpoints",
    "losses": "outputs/losses",
    "figures": "outputs/figures",
}
for p in output_paths.values():
    os.makedirs(p, exist_ok=True)

In [101]:
def collate_mil_with_tab(batch):
    bags, labels, tabs, metas = zip(*batch)
    if tabs[0].numel() == 0:
        tab_batch = torch.zeros(len(batch), 0, dtype=torch.float32)
    else:
        tab_batch = torch.stack(tabs, dim=0).float()
    return list(bags), torch.stack(labels), tab_batch, list(metas)


In [103]:
import torch, torch.nn as nn, torch.nn.functional as F

class AttentionPool(nn.Module):
    def __init__(self, D, H=128):
        super().__init__()
        self.V = nn.Linear(D, H)
        self.w = nn.Linear(H, 1, bias=False)
    def forward(self, Hs):  # Hs: [S,D]
        a = self.w(torch.tanh(self.V(Hs))).squeeze(-1)   # [S]
        a = torch.softmax(a, dim=0)
        E = torch.sum(a[:,None]*Hs, dim=0)               # [D]
        return E, a

class FusiliMILFusion(nn.Module):
    def __init__(self, encoder2d: nn.Module, D: int, tab_in: int, num_classes=3, tab_hidden=(64,128)):
        super().__init__()
        self.encoder2d = encoder2d
        self.attn = AttentionPool(D, H=128)
        # tabular tower (Fusili-like)
        self.tab = nn.Sequential(
            nn.Linear(tab_in, tab_hidden[0]), nn.ReLU(),
            nn.Linear(tab_hidden[0], tab_hidden[1]), nn.ReLU(),
        ) if tab_in>0 else nn.Identity()
        tab_out = tab_hidden[1] if tab_in>0 else 0
        self.norm = nn.LayerNorm(D)
        self.drop = nn.Dropout(0.1)
        self.head = nn.Sequential(
            nn.Linear(D+tab_out, 256), nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, slice_bags, tab_batch):
        B = len(slice_bags)
        device = next(self.parameters()).device
        S_lens = [x.shape[0] for x in slice_bags]
        X = torch.cat(slice_bags, dim=0).to(device)    
        Hs = self.encoder2d(X)                          
        assert Hs.dim()==2, "encoder2d must return [N,D] features"

        logits_list, attn_list = [], []
        start = 0
        tab_feats = self.tab(tab_batch.to(device)) if tab_batch.numel() else torch.zeros(B,0, device=device)
        for i,S in enumerate(S_lens):
            H_i = Hs[start:start+S]                   
            H_i = self.drop(self.norm(H_i))
            E_i, a_i = self.attn(H_i)                
            fused = torch.cat([E_i, tab_feats[i]], dim=0) if tab_feats.shape[1]>0 else E_i
            logit = self.head(fused[None,:])          
            logits_list.append(logit)
            attn_list.append(a_i.detach().cpu())
            start += S
        return torch.cat(logits_list, dim=0), attn_list


In [105]:
import torchvision.models as models
base = models.resnet18(weights=None)      
base.fc = nn.Identity()
encoder2d = nn.Sequential(nn.Conv2d(1,3,1), base)   
feature_dim = 512


In [107]:
from torch.utils.data import DataLoader

class MILDataModule:
    def __init__(self, train_df, val_df, img_col, label_col, tab_cols,
                 batch_size=2, num_workers=0, **ds_kwargs):
        self.train_ds = ScanMILDatasetWithTab(train_df, img_col, label_col, tab_cols, **ds_kwargs)
        self.val_ds   = ScanMILDatasetWithTab(val_df,   img_col, label_col, tab_cols, **ds_kwargs)
        self.batch_size = batch_size
        self.num_workers = num_workers

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True,
                          collate_fn=collate_mil_with_tab, num_workers=self.num_workers)
    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False,
                          collate_fn=collate_mil_with_tab, num_workers=self.num_workers)

def prepare_fusili_mil_data(*, train_df, val_df, img_col, label_col, tab_cols,
                            batch_size=2, num_workers=0,
                            target_size=224, slice_stride=2, max_slices=128,
                            clip_percentiles=(0.5,99.5)):
    return MILDataModule(
        train_df, val_df, img_col, label_col, tab_cols,
        batch_size=batch_size, num_workers=num_workers,
        target_size=target_size, slice_stride=slice_stride, max_slices=max_slices,
        clip_percentiles=clip_percentiles
    )


In [111]:
all_tab_cols = clean.columns.tolist()

ban = {"image_path", "prediction_label", "ID"}  
tab_cols = [c for c in all_tab_cols if c not in ban]

for df in (df_train, df_val):
    for c in tab_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df[tab_cols] = df[tab_cols].fillna(0.0)


In [113]:

data_module = prepare_fusili_mil_data(
    train_df=df_train,
    val_df=df_val,
    img_col="image_path",
    label_col="prediction_label",
    tab_cols=tab_cols,
    batch_size=1,       
    num_workers=0,      
    target_size=224,
    slice_stride=2,
    max_slices=128
)
train_loader = data_module.train_dataloader()
val_loader   = data_module.val_dataloader()


In [ ]:
import torch, torch.nn as nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tab_in = len(tab_cols)
model = FusiliMILFusion(encoder2d, D=feature_dim, tab_in=tab_in, num_classes=3).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
crit = nn.CrossEntropyLoss()  

for epoch in range(100):
    model.train()
    for slice_bags, labels, tab_batch, _ in train_loader:
        labels = labels.to(device)
        logits, _ = model(slice_bags, tab_batch)
        loss = crit(logits, labels)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()

    # val
    model.eval(); correct=total=0
    with torch.no_grad():
        for slice_bags, labels, tab_batch, _ in val_loader:
            labels = labels.to(device)
            logits, _ = model(slice_bags, tab_batch)
            pred = logits.argmax(1)
            correct += (pred==labels).sum().item(); total += labels.numel()
    print(f"epoch {epoch} | val acc {correct/total:.3f}")


In [ ]:
counts = clean['prediction_label'].value_counts().sort_index()
inv_freq = (counts.sum() / (len(counts) * counts)).values
class_weights = torch.tensor(inv_freq, dtype=torch.float32)

data_module = prepare_fusion_data(
    prediction_task="multiclass",
    fusion_model=ConcatImageMapsTabularMaps,
    data_paths=data_paths,
    output_paths=output_paths,
    batch_size=8,
    test_size=0.2,
    multiclass_dimensions=3,
    image_downsample_size=(224, 224),
    num_workers=4
)

In [ ]:
# data_module = prepare_fusion_data(
#     prediction_task="multiclass",
#     fusion_model=ConcatImageMapsTabularMaps,
#     data_paths=data_paths,
#     output_paths=output_paths,
#     batch_size=8,
#     test_size=0.2,
#     multiclass_dimensions=3,
#     image_downsample_size=(224, 224),  # default for CNNs
#     num_workers=4
# )

In [13]:
# df_counts = df_numeric['prediction_label'].value_counts()
# weights = df_numeric['prediction_label'].apply(lambda x: 1/class_counts[x]).values
# pos_weight = tensor(weight_for_0)

In [14]:
# import torch
# import pandas as pd

# counts = df_numeric['prediction_label'].value_counts().sort_index()

# inv_freq = (counts.sum() / (len(counts) * counts)).values
# class_weights = torch.tensor(inv_freq, dtype=torch.float)
# print("class_weights [w0,w1,w2]:", class_weights.tolist())

class_weights [w0,w1,w2]: [0.7426303625106812, 0.7554786801338196, 3.032407522201538]


In [ ]:
trained_model = train_and_save_models(
    data_module=data_module,
    fusion_model=ConcatImageMapsTabularMaps,
    training_modifications={
        "accelerator": "cpu",
        "devices": 1,       # number of GPUs
         "loss_params": {
        #     "pos_weight": pos_weight
        "weight": class_weights },
        # "precision": 16,    # mixed precision
    },
    max_epochs=100,
)

In [ ]:
RealsVsPreds.from_final_val_data(trained_model)
ConfusionMatrix.from_final_val_data(trained_model)
plt.show()